In [92]:
import igl
import meshplot as mp
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits import mplot3d
import matplotlib.tri as mtri
from matplotlib.animation import FuncAnimation
from typing import Tuple
from typing import List

%matplotlib widget
def vec2(x: float, y: float) -> np.ndarray:
    '''
    Creates (2,) ndarray
    '''
    return np.array([x, y], dtype=np.float64)


def hat(p: np.ndarray) -> np.ndarray:
    '''
    Rotates 2d vector by pi/2 counterclockwise
    '''
    return vec2(-p[1], p[0])

def compute_triangle_areas(V: np.ndarray, T: np.ndarray) -> np.ndarray:
    '''
    Computes area of each triangle in V, T mesh
    '''
    E = len(T) # no of triangles/elements
    A = np.zeros(E, dtype=np.float64)
    for e in range(E):
        # indecies of vertecies of current triangle
        i, j, k = T[e, :]   
        # coordinates of vertices
        xi, yi = V[i, :]
        xj, yj = V[j, :]
        xk, yk = V[k, :]
        # distances
        dx1 = xk - xj
        dy1 = yk - yj
        dx2 = xi - xj
        dy2 = yi - yj
        # area of current element
        A[e] =  (dx1*dy2 - dx2*dy1 ) / 2.0
    return A


def make_beam_mesh(width: int, height: int, shape: Tuple[int, int]) -> Tuple[np.ndarray, np.ndarray]:
    '''
    Creates a simple rectangular V, T mesh
    '''
    # mesh is centered around (0, 0)
    x0 = -width/2.0
    y0 = -height/2.0
    I, J = shape #resolution
    # distances between cells
    dx = width/float(I)
    dy = height/float(J)

    V = np.zeros(((I + 1)*(J + 1), 2), np.float64)
    # vecrorized double loop
    iy, ix = np.meshgrid(np.arange(J + 1), np.arange(I + 1), indexing='ij')
    V[:, 0] = x0 + ix.ravel()*dx
    V[:, 1] = y0 + iy.ravel()*dy
    
    T = np.zeros((2*I*J,3),dtype=np.int32)
    for j in range(J):
        for i in range(I):
            k00 = (i  ) + (j  )*(I+1)
            k01 = (i+1) + (j  )*(I+1)
            k10 = (i  ) + (j+1)*(I+1)
            k11 = (i+1) + (j+1)*(I+1)
            e = 2*(i + j*I)
            if (i+j+1)%2:
                T[e,  :] = (k00,k01,k11)
                T[e+1,:] = (k00,k11,k10)
            else:
                T[e,  :] = (k10,k00,k01)
                T[e+1,:] = (k10,k01,k11)  
    return V, T

class MeshInfo:
    def __init__(self, V: np.ndarray, T: np.ndarray):
        """
        Initialize object
        """
        self.A = compute_triangle_areas(V, T)
        if np.any(np.where(self.A<=0)):
            raise ValueError('Triangle with negative orientation exist.')
        # bd[e] stores barycenter of e-th element
        self.bc = igl.barycenter(V, T)
        # stores unique edges of mesh (doesn't repeat common edges)
        self.E = igl.edges(T)
        # each row corresponds to a triangle
        # each entry in the row gives the index of the neighboring triangle that shares corresponding edge
        # if edge is on the boundary (i.e. no neighbpur) the entry is set to -1
        self.TT, _ = igl.triangle_triangle_adjacency(T)
        # VT - 1D array with indecies of all triangles adjacent to all vertices, concatenated together
        # NI - 1D array of length num_vertices + 1, where NI[i] and NI[i+1]
        #       give the start and end indices in VT for the triangles adjacent to vertex i
        self.VT, self.NI = igl.vertex_triangle_adjacency(T, len(V))
        # VV[i] stores indecies of adjacent vertecies to i-th vertex
        self.VV = igl.adjacency_list(T)
        # Store V, and T matrices
        self.V = V
        self.T = T
        mask = np.zeros(len(V),dtype=np.int32)
        # setting one for boundary vertecies
        mask[igl.boundary_loop(T)] = 1
        self.boundary_vertex = mask
        
    def get_triangles_of_vertex(self, i: int) -> np.ndarray:
        """
        Returns all triangles that share i-th vertex
        """
        start = self.NI[i]
        end = self.NI[i+1]
        return self.VT[ start:end]
    
    def get_triangles_of_triangle(self, t: int) -> np.ndarray:
        """
        Returns adjacent triangles to t-th triangle
        """
        return self.TT[t]
    
    def get_triangle_center(self, t: int) -> float:
        """
        Barycenter of triangle t
        """
        return self.bc[t]
    
    def get_vertices_of_vertex(self, i: int) -> np.ndarray:
        '''
        Returns vertices adjacent to vertex i
        '''
        return self.VV[i]
    
    def get_triangle_area(self, t: int) -> float:
        """
        Returns area of triangle t
        """
        return self.A[t]
    
    def get_edge_vector_of_triangle(self, t: int, e: int) -> Tuple[np.ndarray, np.ndarray]:
        """
        Returns: edge vector and coordinates of midpoint for edge e of triangle t
        """
        i = self.T[t,e]
        j = self.T[t,(e+1)%3]
        vi = self.V[i,:].flatten()
        vj = self.V[j,:].flatten()
        u = (vj-vi).flatten()
        return u, (vi+vj)/2


    def get_outward_normal_of_triangle(self, t: int, e: int) -> Tuple[np.ndarray, np.ndarray]:
        '''
        Returns: 
        n - unit outward vector
        m - midpoint of edge e of triangle t
        '''
        u, m = self.get_edge_vector_of_triangle(t,e)
        n = -hat(u)/np.linalg.norm(u)
        return n, m
    
    def get_triangle_triangle_segment(self, t: int, e: int) -> Tuple[np.ndarray, np.ndarray]:
        '''
        Computes a vector between two barycenters
        Returns:
        - vector from triangle t to its neighbor across the edge e
        - midpoint of the segment connecting two barycenters
        '''
        n = self.TT[t,e]
        if n>0:
            ct = self.bc[t].flatten()
            cn = self.bc[n].flatten()
            return cn-ct, (cn+ct)/2
        # If no neighbor across the edge then return None, None
        return None, None    

    def get_opposing_triangle(self, t: int, v: int) -> int:
        '''
        input:
            t - index of triangle
            v - index of edge within triangle (0, 1, 2)
        returns:
            index of neighboring triangle
        '''
        e = (v+1)%3
        return self.TT[t,e]
    
    def is_boundary_vertex(self, v: int) -> bool:
        '''
        Checks whether given vertex lies on boundary
        '''
        if self.boundary_vertex[v]==1:
            return True
        return False




def get_cyclic_permuation(v: int, triangle:np.ndarray) -> np.ndarray:
    '''
    Permutes indexes within triangle array in the way that
    vertex v jumps in front
    '''
    i = triangle[0]
    j = triangle[1]
    k = triangle[2]
    if i==v:
        return np.array([i, j, k])
    if j==v:
        return np.array([j, k, i])
    if k==v:
        return np.array([k, i, j])
    return np.array([-1, -1, -1])

def get_ccw_triangle_order(v: int, T: np.ndarray) -> np.ndarray:
    '''
    Orders set of triangles having a common vertex v in ccw order
    input:
    v - vertex index
    T - (n, 3) array containing n triangles all sharing vertex v
    returns: array of triangle indices ordered ccw around v
    '''
    for t in range(len(T)):
        T[t,:] = get_cyclic_permuation(v, T[t,:])    
    if len(T) == 1:
        ccw_order = np.array([0], dtype=np.int32).flatten()
    else:
        ccw_order = []
        next_order = -np.ones((len(T), ), dtype=np.int32)
        prev_order = -np.ones((len(T), ), dtype=np.int32)
        for s in range(len(T)):
            A = T[s,:]
            for t in range(len(T)):
                B = T[t,:]
                if A[2]==B[1]:
                    next_order[s] = t
                if A[1]==B[2]:
                    prev_order[s] = t
        if np.any(prev_order<0):
            i = np.where(prev_order<0)[0]
            while i>=0:
                ccw_order.append(i)
                i = next_order[i]
        else:
            i = 0
            ccw_order.append(i)
            i = next_order[i]
            while i>0:
                ccw_order.append(i)
                i = next_order[i]            
        ccw_order = np.array(ccw_order, dtype=np.int32).flatten()
    return ccw_order
    
class VertexControlVolume:
    def __init__(self, v: int, info: MeshInfo):
        self.info = info
        self.v = v
        self.c = info.V[v,:].flatten()
        # triangles that get 
        self.labels = info.get_triangles_of_vertex(v)
        K = len(self.labels)
        self.K = K     # no of adjacent triangles
        self.boundary = info.is_boundary_vertex(v)
        T = info.T[self.labels,:]

        for t in range(len(T)):
            T[t,:] = get_cyclic_permuation(v, T[t,:])

        self.Lij = np.zeros(( K, ), dtype=np.float64 )
        self.Lik = np.zeros(( K, ), dtype=np.float64 )
        self.Nij = np.zeros(( K, 2), dtype=np.float64 )
        self.Nik = np.zeros(( K, 2), dtype=np.float64 )
        self.Sij = np.zeros(( K, 2), dtype=np.float64 )
        self.Sik = np.zeros(( K, 2), dtype=np.float64 )
        self.P = np.zeros(( K, 2), dtype=np.float64 )
        self.A = np.sum(info.A[self.labels])/3.0
        for e in range(K):
            i, j, k = T[e, :]
            
            pi = info.V[i,:]
            pj = info.V[j,:]
            pk = info.V[k,:]
            
            Eij = (pj-pi)
            Lij = np.linalg.norm(Eij)
            Nij = -hat(Eij)/Lij
            
            Eik = (pk-pi)
            Lik = np.linalg.norm(Eik)
            Nik = hat(Eik)/Lik
        
            self.Lij[e] = Lij/2
            self.Lik[e] = Lik/2
            self.Nij[e,:] = Nij
            self.Nik[e,:] = Nik
            self.Sij[e,:] = Eij/2
            self.Sik[e,:] = Eik/2
            self.P[e,:] = (pi+pj+pk)/3



In [93]:
# Parameters structure
class Parameters:
    
    def __init__(self, material='default'):
        library = { 'cartilage':(0.69e6, 0.018, 1000),
               'cortical bone': (16.16e9, 0.33, 1600),
               'cancellous bone': (452e6, 0.3, 1600),
               'rubber': (0.01e9, 0.48, 1050),
               'concrete': (30e9, 0.20, 2320),
               'copper': (125e9, 0.35, 8900),
               'steel': (210e9, 0.31, 7800),
               'aluminium': (72e9, 0.34, 2700),
               'glass': ( 50e9,  0.18, 2190),
               'default': (10e5,0.3,1000)}
        E, nu, rho = library[material]
        self.E = E              # Young Modulus
        self.nu = nu            # Poisson ratio
        self.rho = rho          # Mass density
        self.lambda_ =  (E*nu) / ((1+nu)*(1-2*nu))  # Lame 1st
        self.mu_     =  E/(2*(1+nu))                # Lame 2nd

### Solver (I hope)

In [94]:
width = 6.0
height = 2.0
resolution = (12, 4)
pars = Parameters('default') # Change argument to get some funky ass parameters
lamb = pars.lambda_
mu = pars.mu_
rho = pars.rho
g = vec2(0, -9.81)

V, T = make_beam_mesh(6.0, 2.0, (12,4))
nV = len(V)
nT = len(T)

info = MeshInfo(V, T)
CVs = [VertexControlVolume(v, info) for v in range(nV)]

# TODO: iterate over all triangles and compute the deformation gradient for each:

def compute_P(V:np.ndarray, T:np.ndarray, V0: np.ndarray, parameters: Parameters) -> np.ndarray:
    '''
    Computes deformation gradient for each triangle given
    current mesh V, T and initial positions of vertices V0
    '''
    lamb = pars.lambda_
    mu = pars.mu_
    nT = len(T)
    P = np.zeros((nT, 2, 2))
    for e in range(nT):
        i, j, k = T[e, :]
        pi = V[i, :]
        pj = V[j, :]
        pk = V[k, :]
        
        pi0 = V0[i, :]
        pj0 = V0[j, :]
        pk0 = V0[k, :]

        gij = pj - pi
        gik = pk - pi

        gij0 = pj0 - pi0
        gik0 = pk0 - pi0

        D = np.vstack((gij, gik))
        D0 = np.vstack((gij0, gik0))

        F = D@np.linalg.inv(D0)
        
        E = 0.5*(F.T@F - np.eye(2))
        S = lamb*np.trace(E)*np.eye(2) + 2*mu*E

        P[e, :, :] = F@S

    return P
tstop = 10
dt = 0.1
nt = int(tstop/dt)
t_indecies = np.linspace(0, tstop, nt)
print(nt)
x = np.zeros((nt, nV, 2))
velo = np.zeros((nt, nV, 2))
# initial mesh layout
x[0, :, :] = np.copy(V)


for t in range(nt - 1):
    nx, ny = resolution
    V_cur = x[t]
    fv = np.zeros((nV, 2))
    fb = np.zeros((nV, 2))
    ft = np.zeros((nV, 2))
    f_total = np.zeros((nV, 2))
    traction_count = 0

    P = compute_P(V_cur, T, V, pars)
    for v in range(nV):
        current_cv = CVs[v]
        fb[v, :] += current_cv.A*g
        # current vertex is boundary
        if current_cv.boundary:
            # TODO: check if this is free or non-free boundary and compute traction force
            if V[v, 1] == nx - 1:
                
                ft[v, :] = vec2(-0.1, -0.1)
                # traction_count += 1


        triangles = current_cv.labels
        k = current_cv.K
        
        # f = np.zeros((k, 2, 2))
        
        for e in range(k):
            l_ij = current_cv.Lij[e]
            l_ik = current_cv.Lik[e]
            n_ij = current_cv.Nij[e]
            n_ik = current_cv.Nik[e]
            fv[v, :] += -0.5*(P[e]@n_ij*l_ij + P[e]@n_ik*l_ik)
        # total force
        f_total[v, :] = fv[v] + fb[v] + ft[v]
        # time update
        m = rho*current_cv.A
        velo[t + 1, v, :] = velo[t, v, :] + dt/m*f_total[v]
        x[t + 1, v, :] = x[t, v, :] + dt*velo[t + 1, v, :]
        
    
        






        




100


c:\Users\przyb\anaconda3\envs\cmis\lib\site-packages\ipykernel_launcher.py:49: RuntimeWarning: overflow encountered in matmul
c:\Users\przyb\anaconda3\envs\cmis\lib\site-packages\ipykernel_launcher.py:50: RuntimeWarning: invalid value encountered in multiply
c:\Users\przyb\anaconda3\envs\cmis\lib\site-packages\ipykernel_launcher.py:52: RuntimeWarning: invalid value encountered in matmul


In [96]:
class Solver:
    def __init__(self, material: Parameters, width: int=6, height: int=2, resolution: Tuple[int, int]=(12, 6)):
        '''
        Initializing mesh at time 0, and all important parameters
        '''
        # arguments to object variables:
        self.material = material
        self.lamb = material.lambda_
        self.mu = material.mu_
        self.rho = material.rho
        self.width = width
        self.height = height
        self.resolution = resolution
        # creating a mesh
        Vinit, T = make_beam_mesh(width, height, resolution)
        self.Vinit = Vinit
        self.T = T
        # setting an external and traction forces (0 by default, can be changed using method)
        self.b = vec2(0, 0)
        self.traction = vec2(0, 0)
        self.info = MeshInfo(Vinit, T)
        nV = len(V)
        self.CVs = [VertexControlVolume(v, self.info) for v in range(nV)]
        # initially x_t is set to none until simulate method isn't run
        self.x_t = None
        

    def set_gravity(self, x: float=0.0, y: float=0.0):
        self.b = vec2(x, y)
    def set_traction(self, x=0, y=0):
        self.traction = vec2(x, y)/resolution[1]    # normalized to the number of nodes in y direction
    
    def compute_P(self, V:np.ndarray) -> np.ndarray:
        '''
        Computes deformation gradient for each triangle given
        current mesh V, T and initial positions of vertices V0
        '''
        T = self.T
        V0 = self.Vinit
        nT = len(T)
        lamb = self.lamb
        mu = self.mu
        P = np.zeros((nT, 2, 2))
        for e in range(nT):
            i, j, k = T[e, :]
            pi = V[i, :]
            pj = V[j, :]
            pk = V[k, :]
            
            pi0 = V0[i, :]
            pj0 = V0[j, :]
            pk0 = V0[k, :]

            gij = pj - pi
            gik = pk - pi

            gij0 = pj0 - pi0
            gik0 = pk0 - pi0

            D = np.vstack((gij, gik))
            D0 = np.vstack((gij0, gik0))

            F = D@np.linalg.inv(D0)
            
            E = 0.5*(F.T@F - np.eye(2))
            S = lamb*np.trace(E)*np.eye(2) + 2*mu*E

            P[e, :, :] = F@S

        return P

    def simulate(self, t_max: float=10, dt: float=0.1) -> np.ndarray:
        
        Vinit = self.Vinit
        T = self.T
        nV = len(Vinit) # number of vertexes
        nT = len(T) # number of triangles
        max_x_idx = self.resolution[0] - 1
        max_y_idx = self.resolution[1] - 1

        nt = int(t_max/dt)     
        t_idxs = np.linspace(0, t_max, nt)
        # initializing matrices
        x_t = np.zeros((nt, nV, 2))  # Positions of vertices for different times
        v_t = np.zeros((nt, nV, 2))  # Velocities --||--
        x_t[0] = Vinit
        # simulation loop:
        for t_idx in range(nt - 1):
            V_cur = x_t[t_idx]
            stress_P = self.compute_P(V_cur)
            f_tot = np.zeros((nV, 2))
            current_info = MeshInfo(V_cur, self.T)
            right_indices = np.where(self.Vinit[:, 0] > width/2 - 1e-03) # indexes of right boundary

            for v_idx in range(nV):
                
                CV = VertexControlVolume(v_idx, current_info)
                
                
                T_adj = CV.labels
                K = CV.K
                # applying gravity
                
                f_tot[v_idx] += self.b*CV.A
                

                if np.isin(v_idx, right_indices):
                    f_tot[v_idx] += self.traction


                        
                for e in range(K):
                    l_ij = CV.Lij[e]
                    l_ik = CV.Lik[e]
                    n_ij = CV.Nij[e]
                    n_ik = CV.Nik[e]
                    current_P = stress_P[T_adj[e]]
                    f_tot[v_idx] += -0.5*(current_P@n_ij*l_ij + current_P@n_ik*l_ik)
                # velocity update
                mass = self.rho*CV.A
                v_t[t_idx + 1, v_idx] = v_t[t_idx, v_idx] + dt/mass*f_tot[v_idx]
                # position update
                x_t[t_idx + 1, v_idx] = v_t[t_idx, v_idx] + dt*v_t[t_idx + 1, v_idx]
            
        self.x_t = x_t
        return x_t

    def animate(self, interval: int=50):
        ''''
        Animates the mesh over time using matplotlib.
        x: (nt, nV, 2) array of vertex positions over time
        T: (nT, 3) array of triangle indices
        interval: delay between frames in ms
        '''
        x = self.x_t
        T = self.T
        nt = x.shape[0]
        fig, ax = plt.subplots()
        ax.set_aspect('equal')
        min_xy = np.nanmin(x[0], axis=0)
        max_xy = np.nanmax(x[0], axis=0)
        ax.set_xlim(min_xy[0] - 0.5, max_xy[0] + 0.5)
        ax.set_ylim(min_xy[1] - 0.5, max_xy[1] + 0.5)

        # Initial plot
        mesh = ax.triplot(x[0, :, 0], x[0, :, 1], T, color='b')[0]

        def update(frame):
            mesh.set_data(x[frame, :, 0], x[frame, :, 1])
            ax.set_title(f"Time step: {frame}")
            return mesh,

        ani = FuncAnimation(fig, update, frames=nt, interval=interval, blit=False)
        plt.show()
        return ani



test_material = Parameters('default')
solve = Solver(test_material)
solve.set_gravity(0, 0)
solve.set_traction(0, 0)
x_t = solve.simulate()


ValueError: Triangle with negative orientation exist.

In [ ]:
# solve.animate(10)